# Step 3 — Spatial Branch (EfficientNet-B0)
By the end of this notebook you will have:
- A working EfficientNet-B0 spatial branch running on your RTX 5060
- Confirmed output shapes for both feature extraction and standalone modes
- A baseline accuracy number from the spatial-only classifier
- Activation visualizations showing what the backbone attends to

## 3.1 Imports

In [ ]:
import os, sys
from pathlib import Path

# Always anchor to project root
PROJECT_ROOT = Path('D:/cpe646-deepfake')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import yaml

from models.spatial_branch import SpatialBranch
from dataset import get_dataloaders

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Root   : {os.getcwd()}')
print(f'Device : {device}')
print('Imports OK')

## 3.2 Build the spatial branch & inspect architecture

In [ ]:
model = SpatialBranch(
    pretrained=True,
    feature_dim=256,
    dropout=cfg['model']['dropout'],
    extract_features=True       # fusion mode
).to(device)

params = model.count_parameters()
print(f'Total parameters    : {params["total"]:,}')
print(f'Trainable parameters: {params["trainable"]:,}')
print(f'Feature dim output  : {model.feature_dim}')
print(f'\nBackbone: {cfg["model"]["spatial_backbone"]}')

## 3.3 Forward pass — confirm shapes

In [ ]:
model.eval()
x = torch.randn(4, 3, 224, 224).to(device)  # batch of 4 face crops

with torch.no_grad():
    features = model(x)

print(f'Input shape   : {x.shape}        (B, C, H, W)')
print(f'Output shape  : {features.shape}  (B, feature_dim)')
print(f'Output device : {features.device}')
print(f'Output range  : [{features.min():.3f}, {features.max():.3f}]')

# VRAM usage
alloc = torch.cuda.memory_allocated() / 1024**2
print(f'\nVRAM used     : {alloc:.1f} MB')

## 3.4 Standalone classifier mode — get a baseline accuracy

In [ ]:
# Build standalone classifier (for spatial-only ablation baseline)
spatial_clf = SpatialBranch(
    pretrained=True,
    feature_dim=256,
    dropout=cfg['model']['dropout'],
    extract_features=False      # standalone mode → returns logits
).to(device)

# Get data
train_loader, val_loader, _ = get_dataloaders(cfg, use_dummy=True)

# Quick training loop — 3 epochs on dummy data to confirm learning
optimizer = torch.optim.Adam(spatial_clf.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print('Training spatial-only model (3 epochs on dummy data)...')
print('-' * 45)

train_losses, val_accs = [], []

for epoch in range(3):
    # ── Train ──
    spatial_clf.train()
    epoch_loss = 0
    for batch in train_loader:
        spatial = batch['spatial'].to(device)
        labels  = batch['label'].to(device)
        optimizer.zero_grad()
        logits  = spatial_clf(spatial)
        loss    = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # ── Validate ──
    spatial_clf.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            spatial = batch['spatial'].to(device)
            labels  = batch['label'].to(device)
            preds   = spatial_clf(spatial).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    avg_loss = epoch_loss / len(train_loader)
    val_acc  = correct / total
    train_losses.append(avg_loss)
    val_accs.append(val_acc)
    print(f'  Epoch {epoch+1}/3 | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.3f}')

print('-' * 45)
print('Note: dummy data has no real signal — expect ~50% accuracy.')
print('Real accuracy will be meaningful once FF++ / DFDC data is loaded.')

## 3.5 Plot training curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.plot(range(1, 4), train_losses, 'o-', color='#4A90D9')
ax1.set_title('Training Loss (Spatial Branch)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(alpha=0.3)

ax2.plot(range(1, 4), val_accs, 'o-', color='#E05A5A')
ax2.set_title('Validation Accuracy (Spatial Branch)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1)
ax2.axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='Random baseline')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/figures/spatial_branch_training.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/figures/spatial_branch_training.png')

## 3.6 Feature space visualization (PCA on extracted features)

In [ ]:
from sklearn.decomposition import PCA

# Extract features from the whole val set
feat_model = SpatialBranch(pretrained=True, feature_dim=256,
                            extract_features=True).to(device)
feat_model.eval()

all_feats, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        feats = feat_model(batch['spatial'].to(device))
        all_feats.append(feats.cpu().numpy())
        all_labels.extend(batch['label'].numpy())

all_feats  = np.vstack(all_feats)
all_labels = np.array(all_labels)

# PCA to 2D
pca    = PCA(n_components=2)
feats2d = pca.fit_transform(all_feats)

fig, ax = plt.subplots(figsize=(6, 5))
colors = {0: '#4A90D9', 1: '#E05A5A'}
for label, name in [(0, 'Real'), (1, 'Fake')]:
    mask = all_labels == label
    ax.scatter(feats2d[mask, 0], feats2d[mask, 1],
               c=colors[label], label=name, alpha=0.6, s=30)

ax.set_title('Spatial Branch Feature Space (PCA 2D)\n'
             'Note: dummy data — clusters will be meaningful with real data')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/spatial_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/figures/spatial_pca.png')
print('\nWith real data, real and fake clusters should be more separated.')

## 3.7 Save spatial-only baseline checkpoint

In [ ]:
import os
os.makedirs('../checkpoints', exist_ok=True)

torch.save({
    'model_state':     spatial_clf.state_dict(),
    'model_config':    {'feature_dim': 256, 'dropout': 0.3,
                        'extract_features': False},
    'val_acc':         val_accs[-1],
    'train_loss':      train_losses[-1],
    'note':            'spatial-only baseline on dummy data'
}, '../checkpoints/spatial_only_dummy.pt')

print('Checkpoint saved to checkpoints/spatial_only_dummy.pt')
print('\nThis will be retrained on real data in the ablation study.')

---
## Step 3 Complete
- `models/spatial_branch.py` built and tested
- EfficientNet-B0 outputs (B, 256) feature vectors on RTX 5060
- Standalone classifier mode confirmed for ablation study
- Baseline training loop verified
- Feature space and training curves saved to `results/figures/`

**Next → Step 4: Frequency Branch (FFT CNN)**